# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.04992439 -0.59955821 -0.83305189 -0.21990281 -0.03379443]
 [-0.77807947  0.34369146 -0.41926369  0.74783492 -0.39113747]
 [-0.07795425  0.38078175 -0.85353847 -0.66272095 -0.81536503]
 [-0.98134876 -0.4867891  -0.40525472 -0.56432004 -0.80064174]
 [-0.47734429  0.44150709 -0.47992043  0.98489268 -0.24146389]
 [ 0.29420379 -0.68713955 -0.7217168   0.22896116  0.19889333]
 [ 0.82784996 -0.18565076  0.12223982 -0.76831884  0.54054031]
 [-0.67135045  0.6587954  -0.16728375 -0.8742701   0.27803566]
 [ 0.25691013  0.57926096 -0.85871901  0.25399136 -0.80555088]
 [ 0.67244594 -0.97570686 -0.08624593 -0.44501373  0.98615165]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a2', 'a2', 'a2', 'a2', 'a1', 'a1', 'a2', 'a2', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 1, 1, 1, 0, 1, 1, 0, 1, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:36,  1.12s/it]

SVI:   3%|▎         | 1/34 [00:01<00:36,  1.12s/it, loss=1812.2858]

SVI:   6%|▌         | 2/34 [00:01<00:35,  1.12s/it, loss=1609.9547]

SVI:   9%|▉         | 3/34 [00:01<00:34,  1.12s/it, loss=1648.4120]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.17it/s, loss=1411.2397]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.17it/s, loss=1487.7280]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.17it/s, loss=1401.8160]

SVI: 100%|██████████| 34/34 [00:01<00:00, 23.43it/s, loss=1401.8160]

SVI: 100%|██████████| 34/34 [00:01<00:00, 23.43it/s, loss=1589.3044]